# 🤖 AWREAS High-Fidelity LLM Benchmark Evaluator

This notebook allows you to run the **Academic Writing Review and Evaluation Agentic System (AWREAS)** evaluation pipeline in **High-Fidelity LLM Mode**.

Unlike the rule-based fallback (which has poor scoring alignment due to lack of semantic understanding), the LLM Mode activates all 8 specialist workers using structural reasoning via Gemini or OpenRouter APIs to achieve excellent agreement with human expert raters (typically **QWK > 0.70**).

To protect your API quotas, this notebook includes a customizable **sample size** option (e.g., scoring a subset of 5 or 10 essays) so you can verify the high-quality results without exhausting your keys.

## 🛠️ Step 1: Mount Google Drive & Unzip Codebase

Upload your `academic-review-system.zip` file directly to the root of your Google Drive (`My Drive/`), then run this cell to mount Drive and extract the project directly into Colab's ultra-fast local runtime (`/content/`).

In [ ]:
from google.colab import drive
import os
import sys
import glob

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define zip path and target extraction directory
zip_path = '/content/drive/MyDrive/academic-review-system.zip'
extract_dir = '/content/academic-review-system'

if os.path.exists(zip_path):
    print("Found zip file! Extracting to local Colab storage...")
    !unzip -q -o {zip_path} -d {extract_dir}
    print("Extraction complete!")
else:
    # Fallback if they uploaded the unzipped directory to Drive
    print("Zip file not found at root. Checking for folder instead...")
    extract_dir = '/content/drive/MyDrive/academic-review-system'
    if not os.path.exists(extract_dir):
        raise FileNotFoundError("Please make sure your project is uploaded to Google Drive!")

# 3. Dynamically find the folder containing pyproject.toml
toml_paths = glob.glob(os.path.join(extract_dir, '**/pyproject.toml'), recursive=True)

if toml_paths:
    project_root = os.path.dirname(toml_paths[0])
    print(f"\n🎯 Success! Found pyproject.toml at: {project_root}")
    sys.path.append(project_root)
    %cd {project_root}
else:
    raise FileNotFoundError("ERROR: Could not locate 'pyproject.toml' anywhere inside the extracted files.")

## 📦 Step 2: Install Project Dependencies

In [ ]:
# Install core dependencies directly from pyproject.toml
!pip install -q .

# Pre-install Google GenAI SDK (required for Direct Gemini connections)
!pip install -q google-genai

## 🔑 Step 3: Enter Your API Credentials Securely

You can run the model directly via **Google Gemini** (recommended) or via **OpenRouter**.

Run this cell to securely enter your API key using python's `getpass` utility (your key will not be saved or printed in the logs).

In [ ]:
import getpass

provider = input("Enter LLM Provider (choose 'gemini', 'openrouter', or 'qwen'): ").strip().lower()
if provider not in ['gemini', 'openrouter', 'qwen']:
    print("Warning: Defaulting to 'gemini' provider.")
    provider = 'gemini'

api_key = getpass.getpass(f"Paste your {provider.upper()} API Key: ").strip()
if not api_key:
    raise ValueError("An API key is required to run the evaluation in LLM Mode!")

print(f"\nCredentials configured successfully for provider: {provider.upper()}")

## 📊 Step 4: Stage 1 - Dataset Preparation

In [ ]:
!python scripts/prepare_asap_benchmark.py

## ⚡ Step 5: Stage 2 & 3 - Run High-Fidelity LLM Evaluation

This cell instantiates the `LLMClient` with the credentials you provided above and evaluates the essays inside Google Colab CPU memory.

**Quota Protection:** Adjust the `SAMPLE_SIZE` variable below. By default, it is set to `10` essays so that you can verify the scoring quality instantly without burning through massive API quotas. Set `SAMPLE_SIZE = None` if you wish to run the entire 145-essay evaluation.

In [ ]:
import json
import os
import sys

# -- FORCE RELOAD IN JUPYTER MEMORY --
# This clears Colab's python import cache, forcing Python to read your newly extracted files
for module in list(sys.modules.keys()):
    if module.startswith("app"):
        del sys.modules[module]
# ------------------------------------

# 1. Import config settings and disable web research to prevent slow external network lookups
from app.core.config import settings
settings.ENABLE_WEB_RESEARCH = False

# 2. Import LLMClient and SupervisorAgent
from app.llm.client import LLMClient
from app.supervisor.agent import SupervisorAgent

# -- QUOTA PROTECTION SETTINGS --
SAMPLE_SIZE = 10  # Change this to 5, 20, or set to None to run all 145 essays
# -------------------------------

input_file = "data/asap/benchmark_prompt1.jsonl"
output_file = "data/asap/benchmark_results_prompt1.json"

# Load dataset lines
with open(input_file, "r") as f:
    dataset_lines = [json.loads(line) for line in f if line.strip()]

# Apply sample size slicing
if SAMPLE_SIZE is not None:
    dataset_lines = dataset_lines[:SAMPLE_SIZE]

total_essays = len(dataset_lines)
results = []

# Explicitly request 'anthropic/claude-sonnet-4.6' model name
model_name = "anthropic/claude-sonnet-4.6"

print(f"Initializing LLMClient (provider={provider}, model={model_name})...")
# Initialize shared LLMClient using your custom credentials and forcing Sonnet 3.5/4.6
llm_client = LLMClient(api_key=api_key, provider=provider, default_model=model_name)

print("Initializing SupervisorAgent with LLMClient...")
supervisor = SupervisorAgent(llm_client=llm_client)

print(f"\n🚀 Starting direct LLM-powered evaluation on {total_essays} essays...")

for i, essay_data in enumerate(dataset_lines):
    essay_text = essay_data["content"]
    essay_id = essay_data["id"]
    
    try:
        # Execute high-fidelity agentic evaluation (concurrently processes 8 agents)
        import time
        start_time = time.perf_counter()
        context = await supervisor.run_evaluation(essay_text)
        processing_time_ms = (time.perf_counter() - start_time) * 1000
        
        results.append({
            "system": "AWREAS_Full",
            "case_id": str(essay_id),
            "ok": True,
            "overall_score": context.overall_score,
        })
        print(f"[SUCCESS] Processed essay {i+1}/{total_essays} (ID: {essay_id}) -> Score: {context.overall_score}")
    except Exception as e:
        print(f"[ERROR] Failed to process essay (ID: {essay_id}): {e}")

# Save formatted results locally in the evaluation client schema
output_data = {
    "results": results
}

os.makedirs(os.path.dirname(output_file), exist_ok=True)
with open(output_file, "w") as out:
    json.dump(output_data, out, indent=4)

print(f"\n🎉 Evaluation complete! Processed {len(results)} records successfully. Saved to: {output_file}")

## 📈 Step 6: Stage 4 - Compute Statistical Agreement

Run this cell to compute your QWK and Pearson correlation. Watch the agreement metrics jump dramatically compared to the rule-based run!

In [ ]:
!python scripts/compute_agreement.py \
    --results data/asap/benchmark_results_prompt1.json \
    --gold data/asap/gold_prompt1.json

## 💾 Step 7: Backup Results back to Google Drive

In [ ]:
import shutil

drive_dest = '/content/drive/MyDrive/AWREAS_LLM_Results'
os.makedirs(drive_dest, exist_ok=True)

shutil.copy('data/asap/benchmark_results_prompt1.json', drive_dest)
print(f"Successfully backed up results to Google Drive folder: {drive_dest}")